# ASL (English) Sign Language - YOLOv5 Training
Train a YOLOv5 model to detect ASL fingerspelling letters (A-Z + SPACE).

**Features:**
- Downloads dataset from Kaggle automatically
- Auto-generates YOLO bounding boxes using MediaPipe
- Saves checkpoints to Google Drive (survives runtime disconnects)
- Can resume training from last checkpoint

**Before you start:**
1. Set runtime to GPU: Runtime > Change runtime type > T4 GPU
2. Have your `kaggle.json` API key file ready (from kaggle.com/settings)

## Step 1: Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ArSL_Project'
DATASET_DIR = os.path.join(PROJECT_DIR, 'datasets_english')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints_english')
RAW_DIR = '/content/asl_raw'  # local to Colab (faster I/O)

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

# Pin mediapipe to 0.10.14 — newer versions removed the solutions API
!pip install ultralytics kaggle mediapipe==0.10.14 -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify mediapipe solutions API is available
import mediapipe as mp
print(f'MediaPipe: {mp.__version__}')
assert hasattr(mp, 'solutions'), 'ERROR: mediapipe.solutions not found. Re-run this cell.'
print('MediaPipe solutions API: OK')

## Step 2: Download ASL Dataset from Kaggle
Upload your `kaggle.json` file when prompted. You can get it from https://www.kaggle.com/settings > API > Create New Token

In [ ]:
import os

# Check if dataset already extracted
EXTRACTED_DIR = os.path.join(RAW_DIR, 'asl_alphabet_train', 'asl_alphabet_train')

if os.path.isdir(EXTRACTED_DIR) and len(os.listdir(EXTRACTED_DIR)) >= 27:
    print('Dataset already downloaded and extracted! Skipping download.')
    print(f'Found {len(os.listdir(EXTRACTED_DIR))} folders in {EXTRACTED_DIR}')
else:
    # Upload kaggle.json
    from google.colab import files
    print('Please upload your kaggle.json file:')
    uploaded = files.upload()

    # Set up Kaggle credentials
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    # Download the ASL alphabet dataset
    print('\nDownloading ASL Alphabet dataset from Kaggle...')
    !kaggle datasets download -d grassknoted/asl-alphabet -p {RAW_DIR} --unzip

    print(f'\nExtracted to: {RAW_DIR}')
    print(f'Folders: {os.listdir(os.path.join(RAW_DIR, "asl_alphabet_train", "asl_alphabet_train"))}')

## Step 3: Generate YOLO Bounding Box Labels with MediaPipe
This processes all images through MediaPipe hand detection to create bounding box labels.
Takes ~15-30 minutes. Progress is saved per-class — safe to re-run if interrupted.

In [ ]:
import cv2
import mediapipe as mp
import random
import shutil

# 27 classes: A-Z + SPACE (skip 'del' and 'nothing')
CLASS_NAMES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','SPACE'
]

FOLDER_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
EXTRACTED_DIR = os.path.join(RAW_DIR, 'asl_alphabet_train', 'asl_alphabet_train')

# Output directories (on Drive for persistence)
TRAIN_IMG = os.path.join(DATASET_DIR, 'train', 'images')
TRAIN_LBL = os.path.join(DATASET_DIR, 'train', 'labels')
VAL_IMG   = os.path.join(DATASET_DIR, 'valid', 'images')
VAL_LBL   = os.path.join(DATASET_DIR, 'valid', 'labels')

for d in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(d, exist_ok=True)

# Check if already done
existing_train = len([f for f in os.listdir(TRAIN_IMG) if f.endswith(('.jpg','.jpeg','.png'))]) if os.path.isdir(TRAIN_IMG) else 0
existing_val   = len([f for f in os.listdir(VAL_IMG)   if f.endswith(('.jpg','.jpeg','.png'))]) if os.path.isdir(VAL_IMG)   else 0

if existing_train > 60000 and existing_val > 15000:
    print(f'Dataset already prepared! Train: {existing_train}, Val: {existing_val}')
    print('Skipping bbox generation. Delete datasets_english/ on Drive to re-run.')
else:
    print('Generating YOLO bounding boxes with MediaPipe...')
    print('This takes ~15-30 minutes. Progress is saved per-class.\n')

    random.seed(42)
    total_processed = 0
    total_fallback  = 0

    # Use solutions.hands (requires mediapipe==0.10.14)
    mp_hands = mp.solutions.hands

    for class_name, class_id in FOLDER_MAP.items():
        folder_name = class_name.lower() if class_name != 'SPACE' else 'space'
        folder_path = os.path.join(EXTRACTED_DIR, folder_name)
        if not os.path.isdir(folder_path):
            folder_path = os.path.join(EXTRACTED_DIR, folder_name.upper())
        if not os.path.isdir(folder_path):
            print(f'  WARNING: Folder not found for {class_name}, skipping')
            continue

        images = sorted([f for f in os.listdir(folder_path)
                         if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])

        # Skip this class if already done (resume support)
        expected_train = int(len(images) * 0.8)
        existing_for_class = len([f for f in os.listdir(TRAIN_IMG)
                                   if f.startswith(f'{class_name}_')]) if os.path.isdir(TRAIN_IMG) else 0
        if existing_for_class >= expected_train:
            print(f'  {class_name}: already done ({existing_for_class} train images), skipping')
            total_processed += len(images)
            continue

        # 80/20 split
        random.shuffle(images)
        split_idx  = int(len(images) * 0.8)
        train_imgs = images[:split_idx]
        val_imgs   = images[split_idx:]

        class_fallback = 0

        with mp_hands.Hands(
            static_image_mode=True,
            max_num_hands=1,
            min_detection_confidence=0.3
        ) as hands:
            for split_name, img_list, img_dir, lbl_dir in [
                ('train', train_imgs, TRAIN_IMG, TRAIN_LBL),
                ('val',   val_imgs,   VAL_IMG,   VAL_LBL)
            ]:
                for img_name in img_list:
                    img_path = os.path.join(folder_path, img_name)
                    img = cv2.imread(img_path)
                    if img is None:
                        continue

                    rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    result = hands.process(rgb)

                    if result.multi_hand_landmarks:
                        lms   = result.multi_hand_landmarks[0]
                        xs    = [lm.x for lm in lms.landmark]
                        ys    = [lm.y for lm in lms.landmark]
                        x_min, x_max = min(xs), max(xs)
                        y_min, y_max = min(ys), max(ys)
                        mx = (x_max - x_min) * 0.15
                        my = (y_max - y_min) * 0.15
                        x_min = max(0.0, x_min - mx)
                        y_min = max(0.0, y_min - my)
                        x_max = min(1.0, x_max + mx)
                        y_max = min(1.0, y_max + my)
                        cx  = (x_min + x_max) / 2
                        cy  = (y_min + y_max) / 2
                        bw  = x_max - x_min
                        bh  = y_max - y_min
                        label_line = f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'
                    else:
                        # Fallback: images are tightly cropped, use 80% of frame
                        label_line = f'{class_id} 0.5 0.5 0.8 0.8'
                        class_fallback += 1

                    out_basename = f'{class_name}_{img_name}'
                    lbl_basename = os.path.splitext(out_basename)[0] + '.txt'
                    shutil.copy2(img_path, os.path.join(img_dir, out_basename))
                    with open(os.path.join(lbl_dir, lbl_basename), 'w') as f:
                        f.write(label_line + '\n')

        total_processed += len(images)
        total_fallback  += class_fallback
        print(f'  {class_name}: {len(images)} images (fallback: {class_fallback})')

    print(f'\nTotal: {total_processed} images processed, {total_fallback} fallbacks')

# Final counts
train_count = len([f for f in os.listdir(TRAIN_IMG) if f.endswith(('.jpg','.jpeg','.png'))]) if os.path.isdir(TRAIN_IMG) else 0
val_count   = len([f for f in os.listdir(VAL_IMG)   if f.endswith(('.jpg','.jpeg','.png'))]) if os.path.isdir(VAL_IMG)   else 0
print(f'\nFinal dataset: {train_count} train, {val_count} val images')

## Step 4: Verify Dataset & Create data.yaml

In [ ]:
import yaml

train_imgs = os.path.join(DATASET_DIR, 'train', 'images')
val_imgs   = os.path.join(DATASET_DIR, 'valid', 'images')

for path, name in [(DATASET_DIR + '/train/images', 'Train images'),
                    (DATASET_DIR + '/train/labels', 'Train labels'),
                    (DATASET_DIR + '/valid/images', 'Val images'),
                    (DATASET_DIR + '/valid/labels', 'Val labels')]:
    count  = len(os.listdir(path)) if os.path.exists(path) else 0
    status = 'OK' if count > 0 else 'MISSING!'
    print(f'{name}: {count} files [{status}]')

ASL_CLASSES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','SPACE'
]

yaml_path = '/content/data_asl.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({
        'train': train_imgs,
        'val':   val_imgs,
        'nc':    27,
        'names': ASL_CLASSES
    }, f, default_flow_style=False, allow_unicode=True)

print(f'\ndata_asl.yaml ready!')

## Step 5: Train YOLOv5 (saves to Drive — resumable!)

Training saves checkpoints to Google Drive every 5 epochs.
If the runtime disconnects, just re-run this cell to resume from the last checkpoint.

In [ ]:
from ultralytics import YOLO

# Check if there's a previous checkpoint to resume from
resume_path = os.path.join(CHECKPOINT_DIR, 'weights', 'last.pt')

if os.path.exists(resume_path):
    print('Found previous checkpoint! Resuming training...')
    print(f'Checkpoint: {resume_path}')
    model   = YOLO(resume_path)
    results = model.train(resume=True)
else:
    print('Starting fresh training...')
    model   = YOLO('yolov5s.pt')
    results = model.train(
        data=yaml_path,
        epochs=80,
        imgsz=640,
        batch=32,        # reduce to 16 if you get OOM errors
        device=0,
        patience=15,
        save=True,
        save_period=5,   # checkpoint every 5 epochs to Drive
        project=CHECKPOINT_DIR,
        name='',
        exist_ok=True,
    )

print('\nTraining complete!')

## Step 6: Evaluate Model

In [ ]:
best_pt    = os.path.join(CHECKPOINT_DIR, 'weights', 'best.pt')
best_model = YOLO(best_pt)
metrics    = best_model.val(data=yaml_path)
print(f'\nmAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## Step 7: Export & Download

Download `english_sign_best.pt` and place it in your local project:
```
models/english_sign_best.pt
```

In [ ]:
import shutil

best_pt    = os.path.join(CHECKPOINT_DIR, 'weights', 'best.pt')
final_dest = os.path.join(PROJECT_DIR, 'english_sign_best.pt')
shutil.copy2(best_pt, final_dest)
print(f'Model saved to Drive: {final_dest}')

# Export ONNX
best_model.export(format='onnx')
onnx_src = best_pt.replace('.pt', '.onnx')
if os.path.exists(onnx_src):
    onnx_dest = os.path.join(PROJECT_DIR, 'english_sign_best.onnx')
    shutil.copy2(onnx_src, onnx_dest)
    print(f'ONNX saved to Drive: {onnx_dest}')

print(f'\nDone! Download english_sign_best.pt from your Google Drive.')

In [ ]:
# Optional: Download directly to your browser
from google.colab import files
files.download(final_dest)